# ResQ-AI: Smoke Test Notebook

This notebook runs the complete ResQ-AI pipeline in smoke mode.
It uses synthetic data and completes in under 5 minutes on CPU.

**No GPU required for smoke tests.**

## 1. Setup

In [ ]:
# Clone the repository
!git clone https://github.com/resq-ai/resq-ai.git 2>/dev/null || true
%cd resq-ai

In [ ]:
# Install dependencies
!pip install -q torch torchvision timm numpy scipy pyyaml tqdm pytest \
    scikit-learn matplotlib seaborn

## 2. Verify Installation

In [ ]:
import torch
import numpy as np
print(f'PyTorch: {torch.__version__}')
print(f'NumPy: {np.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 3. Run Smoke Tests

In [ ]:
# Run pytest smoke tests
!python -m pytest tests/test_smoke.py -v -m smoke --tb=short 2>&1 | tail -30

## 4. Quick Training Demo

In [ ]:
import sys
sys.path.insert(0, '.')

from src.data.synthetic import SyntheticFloodDataset, create_synthetic_dataloaders
from src.models.resqnet import ResQNet
from src.models.losses import BCEDiceLoss
from src.utils.seed import set_seed
import torch
import torch.nn.functional as F

# Setup
set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Config
config = {
    'project': {'seed': 42},
    'data': {
        'crop_size': 64,
        'batch_size': 4,
        'num_train': 32,
        'num_val': 8,
        'num_test': 8,
        'modality_dropout': 0.2,
    },
    'model': {
        'sar_channels': 2,
        'optical_channels': 13,
        'geo_channels': 6,
        'rainfall_seq_len': 30,
    },
}

# Create dataloaders
loaders = create_synthetic_dataloaders(config, num_workers=0)
print(f'Train batches: {len(loaders["train"])}')
print(f'Val batches: {len(loaders["valid"])}')

In [ ]:
# Build model
model = ResQNet(
    sar_channels=2,
    optical_channels=13,
    geo_channels=6,
    rainfall_seq_len=30,
    rainfall_d_model=32,
    rainfall_nhead=2,
    rainfall_layers=1,
    encoder_name='resnet34',
    pretrained=False,
    fusion='film_crossattention',
    dropout=0.1,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

# Loss and optimizer
criterion = BCEDiceLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Quick training loop
model.train()
for epoch in range(2):
    epoch_loss = 0
    for batch in loaders['train']:
        sar = batch['sar'].to(device)
        optical = batch['optical'].to(device)
        geo = batch['geo'].to(device)
        rainfall = batch['rainfall'].to(device)
        label = batch['label'].to(device)
        mask = batch['modality_mask'].to(device)
        
        optimizer.zero_grad()
        output = model(sar=sar, optical=optical, geo=geo, rainfall=rainfall, modality_mask=mask)
        logits = output['logits']
        
        # Resize to match label
        if logits.shape[-2:] != label.shape[-2:]:
            logits = F.interpolate(logits, size=label.shape[-2:], mode='bilinear', align_corners=False)
        
        loss = criterion(logits, label.unsqueeze(1))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    print(f'Epoch {epoch+1}/2, Loss: {epoch_loss/len(loaders["train"]):.4f}')

print('Training complete!')

## 5. Evaluation Demo

In [ ]:
from src.evaluation.metrics import FloodMetrics
from src.models.resqnet import mc_dropout_predict

# Evaluate on test set
model.eval()
all_probs = []
all_labels = []

for batch in loaders['test']:
    with torch.no_grad():
        output = model(
            sar=batch['sar'].to(device),
            optical=batch['optical'].to(device),
            geo=batch['geo'].to(device),
            rainfall=batch['rainfall'].to(device),
            modality_mask=batch['modality_mask'].to(device),
        )
        probs = output['probs']
        if probs.shape[-2:] != batch['label'].shape[-2:]:
            probs = F.interpolate(probs, size=batch['label'].shape[-2:], mode='bilinear', align_corners=False)
        all_probs.append(probs.squeeze(1).cpu())
        all_labels.append(batch['label'])

probs_cat = torch.cat(all_probs)
labels_cat = torch.cat(all_labels)

metrics = FloodMetrics.compute(probs_cat, labels_cat)
print('Test Metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

## 6. MC Dropout Uncertainty

In [ ]:
# MC Dropout uncertainty estimation
batch = next(iter(loaders['test']))
result = mc_dropout_predict(
    model, num_samples=5,
    sar=batch['sar'].to(device),
    optical=batch['optical'].to(device),
    geo=batch['geo'].to(device),
    rainfall=batch['rainfall'].to(device),
    modality_mask=batch['modality_mask'].to(device),
)

print(f'Mean prob range: [{result["probs"].min():.3f}, {result["probs"].max():.3f}]')
print(f'Uncertainty (std) range: [{result["std"].min():.3f}, {result["std"].max():.3f}]')
print(f'Epistemic uncertainty range: [{result["epistemic_uncertainty"].min():.3f}, {result["epistemic_uncertainty"].max():.3f}]')

## 7. Conformal Prediction

In [ ]:
from src.uq.conformal import ConformalCalibrator
import numpy as np

# Calibrate on validation set
cal_probs = probs_cat.numpy().flatten()
cal_labels = labels_cat.numpy().flatten()

calibrator = ConformalCalibrator()
calibrator.calibrate(cal_probs, cal_labels, alpha=0.1)
print(f'Conformal threshold: {calibrator.threshold:.4f}')

# Evaluate coverage
test_probs = probs_cat.numpy().flatten()
test_labels = labels_cat.numpy().flatten()
sets = calibrator.predict_sets(test_probs)
coverage = (sets == (test_labels == 1)).mean()
print(f'Empirical coverage: {coverage:.4f}')

## Done!

This notebook demonstrated the core ResQ-AI pipeline:
1. Synthetic data generation
2. ResQNet model training
3. Flood prediction evaluation
4. MC Dropout uncertainty estimation
5. Conformal prediction calibration

For full experiments with real data, see the `experiments/run_experiments.py` script.